# Agent的高级用法-ToolStrategy
## 1、ToolStrategy的多种结构化输出方式：schema参数

### 1.1 Pydantic类型

使用CloseAI平台的模型

In [18]:
from dataclasses import dataclass

from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# 从.env文件中加载环境变量
load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
        extra_body={
        "thinking": {
            "type": "disabled"
        }
    },
)

使用OpenRounter平台的模型

In [19]:
from langchain_deepseek import ChatDeepSeek
from dotenv import load_dotenv
import os

load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

model = ChatDeepSeek(
    model="deepseek-v4-flash",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    extra_body={
        "thinking": {
            "type": "disabled"
        }
    },
)

举例1：

In [20]:
from pydantic import BaseModel, Field
from langchain.agents.structured_output import ToolStrategy
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from rich import print as rprint

class ContactInfo(BaseModel):
    """用户的联系方式"""
    name: str = Field(description="用户姓名")
    email: str = Field(description="用户邮箱地址")
    phone: str = Field(description="用户的手机号")


agent = create_agent(
    model=model,
    response_format=ToolStrategy(schema=ContactInfo)
)

response = agent.invoke({
    "messages" : [
        HumanMessage(content="从这段话中抽取结构化信息：小明的邮箱是shkstart@atguigu.com,电话是：13012341234")
    ]
})


rprint(response)

{
    'messages': [
        HumanMessage(
            content='从这段话中抽取结构化信息：小明的邮箱是shkstart@atguigu.com,电话是：13012341234',
            additional_kwargs={},
            response_metadata={},
            id='79d859cf-e7bb-438f-b2c1-d3d1162f5802'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 78,
                    'prompt_tokens': 350,
                    'total_tokens': 428,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0},
                    'prompt_cache_hit_tokens': 0,
                    'prompt_cache_miss_tokens': 350
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e',
                'id': '24177718-83eb-4513-9dca-8610368cdc9f',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a05c2c-067f-72b0-8f3c-7c194514f7cf-0',
            tool_calls=[
                {
                    'name': 'ContactInfo',
                    'args': {'name': '小明', 'email': 'shkstart@atguigu.com', 'phone': '13012341234'},
                    'id': 'call_00_wo8xXtR53wMvzorNmOcX1875',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 350,
                'output_tokens': 78,
                'total_tokens': 428,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content="Returning structured response: name='小明' email='shkstart@atguigu.com' phone='13012341234'",
            name='ContactInfo',
            id='9a062c0b-181d-4774-87c1-7f8231bdb107',
            tool_call_id='call_00_wo8xXtR53wMvzorNmOcX1875'
        )
    ],
    'structured_response': ContactInfo(name='小明', email='shkstart@atguigu.com', phone='13012341234')
}

举例2：添加工具的调用

In [21]:
from langchain_core.messages import SystemMessage
from pydantic import BaseModel, Field
from typing import Literal
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.tools import tool
from rich import print as rprint

# 定义工具
@tool(parse_docstring=True)
def search_customer_database(query: str) -> str:
    """
    在客户数据库中搜索信息

    Args:
        query (str): 客户查询字符串，例如 "张三" 或 "李四"

    Returns:
        str: 客户记录字符串，包含客户姓名、等级、最近购买日期和累计消费
    """
    # 模拟数据库查询结果
    if "张三" in query.lower():
        return "客户记录：张三，VIP客户，最近购买日期：2026-01-15，累计消费：$15,000"
    elif "李四" in query.lower():
        return "客户记录：李四，普通客户，最近购买日期：2025-12-20，累计消费：$3,200"
    else:
        return f"关于客户{query}，无记录"


@tool(parse_docstring=True)
def send_email(customer: str) -> str:
    """
    发送感谢邮件

    Args:
        customer (str): 客户名称，例如 "张三" 或 "李四"

    Returns:
        str: 确认消息，包含已发送的客户名称
    """
    return f"已向 {customer} 发送感谢邮件"


# 定义Pydantic Schema
class CustomerAnalysis(BaseModel):
    """客户分析报告"""
    customer_name: str = Field(None, description="客户姓名")
    customer_tier: Literal["潜在客户", "普通客户", "VIP客户", "流失风险"] = Field("潜在客户",description="客户等级,只能是潜在客户、普通客户、VIP客户或流失风险")
    recent_activity: str = Field(None, description="最近活动")
    spending_level: Literal["低", "中", "高"] = Field(None, description="消费水平")
    send_email: bool = Field(False, description="是否已发送感谢邮件")


# 创建智能体
agent = create_agent(
    model=model,
    system_prompt=SystemMessage(content=""
                                        "请分析指定客户的情况："
                                        "1. 先搜索客户数据库了解最新情况 "
                                        "2. 如果是VIP客户，则发送感谢邮件 "
                                        "3. 基于搜索结果生成结构化分析报告 "
                                        "4. 如果用户提问与客户记录无关或找不到客户信息，则返回空对象，不发送感谢邮件"
                                ),
    tools=[search_customer_database, send_email],
    response_format=ToolStrategy(schema=CustomerAnalysis)
)

# 执行分析
result = agent.invoke({
    # "messages": [{"role": "user", "content": "请分析客户张三"}]
    "messages": [{"role": "user","content": "请分析客户李四"}]
    # "messages": [{"role": "user","content": "请分析客户王五"}]
    # "messages": [{"role": "user","content": "今天天气如何"}]
})

# 处理结果
rprint(result)

# if "structured_response" in result:
#     analysis = result["structured_response"]
#     print(analysis)

{
    'messages': [
        HumanMessage(
            content='请分析客户李四',
            additional_kwargs={},
            response_metadata={},
            id='3b324ba3-f863-4edf-890a-a5809cc7bf07'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 40,
                    'prompt_tokens': 623,
                    'total_tokens': 663,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0},
                    'prompt_cache_hit_tokens': 0,
                    'prompt_cache_miss_tokens': 623
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e',
                'id': '6e8f5cd2-bc35-4224-a076-142f62ad5a93',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a05c2c-0b84-7083-8122-bad20e61e284-0',
            tool_calls=[
                {
                    'name': 'search_customer_database',
                    'args': {'query': '李四'},
                    'id': 'call_00_3ciealjSTnjp5XUNjkjn1133',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 623,
                'output_tokens': 40,
                'total_tokens': 663,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content='客户记录：李四，普通客户，最近购买日期：2025-12-20，累计消费：$3,200',
            name='search_customer_database',
            id='fa6f5b72-dd18-48c2-ac57-c0085fa78c95',
            tool_call_id='call_00_3ciealjSTnjp5XUNjkjn1133'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 120,
                    'prompt_tokens': 709,
                    'total_tokens': 829,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 640},
                    'prompt_cache_hit_tokens': 640,
                    'prompt_cache_miss_tokens': 69
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e',
                'id': '58a1081a-25d5-43ab-b790-51053d57fecc',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a05c2c-0eb7-72a3-ae97-bf9b407cceb0-0',
            tool_calls=[
                {
                    'name': 'CustomerAnalysis',
                    'args': {
                        'customer_name': '李四',
                        'customer_tier': '普通客户',
                        'recent_activity': '最近购买日期：2025-12-20',
                        'spending_level': '中',
                        'send_email': False
                    },
                    'id': 'call_00_2m22TtXqS3biQfpJA62z5181',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 709,
                'output_tokens': 120,
                'total_tokens': 829,
                'input_token_details': {'cache_read': 640},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content="Returning structured response: customer_name='李四' customer_tier='普通客户' 
recent_activity='最近购买日期：2025-12-20' spending_level='中' send_email=False",
            name='CustomerAnalysis',
       

### 1.2 TypedDict类型

举例1：

In [22]:
from typing import TypedDict, Annotated
from pydantic import BaseModel, Field
from langchain.agents.structured_output import ToolStrategy
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from rich import print as rprint

class ContactInfo(TypedDict):
    """用户的联系方式"""
    name: Annotated[str, ..., "用户姓名"]
    email: Annotated[str, ..., "用户邮箱地址"]
    phone: Annotated[str, ..., "用户的手机号"]

agent = create_agent(
    model=model,
    response_format=ToolStrategy(schema=ContactInfo)
)

response = agent.invoke({
    "messages" : [
        HumanMessage(content="从这段话中抽取结构化信息：小明的邮箱是shkstart@atguigu.com,电话是：13012341234")
    ]
})


rprint(response)

{
    'messages': [
        HumanMessage(
            content='从这段话中抽取结构化信息：小明的邮箱是shkstart@atguigu.com,电话是：13012341234',
            additional_kwargs={},
            response_metadata={},
            id='b0fd7f85-e2dc-4085-b5f9-1353636ab382'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 78,
                    'prompt_tokens': 327,
                    'total_tokens': 405,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0},
                    'prompt_cache_hit_tokens': 0,
                    'prompt_cache_miss_tokens': 327
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e',
                'id': '759b52dd-e1f0-4809-9dab-b7c4ad5a0df6',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a05c2c-13da-7fe2-82b7-44c9e457be5b-0',
            tool_calls=[
                {
                    'name': 'ContactInfo',
                    'args': {'name': '小明', 'email': 'shkstart@atguigu.com', 'phone': '13012341234'},
                    'id': 'call_00_KKY8T5oR7pRzIFxsLzfg3783',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 327,
                'output_tokens': 78,
                'total_tokens': 405,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content="Returning structured response: {'name': '小明', 'email': 'shkstart@atguigu.com', 'phone': 
'13012341234'}",
            name='ContactInfo',
            id='2910f692-436a-4627-8324-4eb894651c78',
            tool_call_id='call_00_KKY8T5oR7pRzIFxsLzfg3783'
        )
    ],
    'structured_response': {'name': '小明', 'email': 'shkstart@atguigu.com', 'phone': '13012341234'}
}

举例2：

In [23]:
from langchain_core.messages import SystemMessage
from pydantic import BaseModel, Field
from typing import Literal, Optional
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.tools import tool
from rich import print as rprint

# 定义工具
@tool(parse_docstring=True)
def search_customer_database(query: str) -> str:
    """
    在客户数据库中搜索信息

    Args:
        query (str): 客户查询字符串，例如 "张三" 或 "李四"

    Returns:
        str: 客户记录字符串，包含客户姓名、等级、最近购买日期和累计消费
    """
    # 模拟数据库查询结果
    if "张三" in query.lower():
        return "客户记录：张三，VIP客户，最近购买日期：2026-01-15，累计消费：$15,000"
    elif "李四" in query.lower():
        return "客户记录：李四，普通客户，最近购买日期：2025-12-20，累计消费：$3,200"
    else:
        return f"关于客户{query}，无记录"


@tool(parse_docstring=True)
def send_email(customer: str) -> str:
    """
    发送感谢邮件

    Args:
        customer (str): 客户名称，例如 "张三" 或 "李四"

    Returns:
        str: 确认消息，包含已发送的客户名称
    """
    return f"已向 {customer} 发送感谢邮件"


# 使用 TypedDict 定义客户分析报告 Schema
class CustomerAnalysis(TypedDict):
    """客户分析报告"""
    customer_name: Annotated[Optional[str], None, "客户姓名"]
    customer_tier: Annotated[Literal["潜在客户", "普通客户", "VIP客户", "流失风险"], "潜在客户", "客户等级"]
    recent_activity: Annotated[Optional[str], None, "最近活动"]
    spending_level: Annotated[Optional[Literal["低", "中", "高"]], None, "消费水平"]
    send_email: Annotated[bool, False, "是否已发送感谢邮件"]

# 创建智能体
agent = create_agent(
    model=model,
    system_prompt=SystemMessage(content=""
                                        "请分析指定客户的情况："
                                        "1. 先搜索客户数据库了解最新情况 "
                                        "2. 如果是VIP客户，则发送感谢邮件 "
                                        "3. 基于搜索结果生成结构化分析报告 "
                                        "4. 如果用户提问与客户记录无关或找不到客户信息，则返回空对象，不发送感谢邮件"
                                ),
    tools=[search_customer_database, send_email],
    response_format=ToolStrategy(schema=CustomerAnalysis)
)

# 执行分析
result = agent.invoke({
    # "messages": [{"role": "user", "content": "请分析客户张三"}]
    "messages": [{"role": "user","content": "请分析客户李四"}]
    # "messages": [{"role": "user","content": "请分析客户王五"}]
    # "messages": [{"role": "user","content": "今天天气如何"}]
})

# 处理结果
rprint(result)

# if "structured_response" in result:
#     analysis = result["structured_response"]
#     print(analysis)

{
    'messages': [
        HumanMessage(
            content='请分析客户李四',
            additional_kwargs={},
            response_metadata={},
            id='9b9e2ffb-d582-4089-93c4-ce457db39fad'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 40,
                    'prompt_tokens': 611,
                    'total_tokens': 651,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0},
                    'prompt_cache_hit_tokens': 0,
                    'prompt_cache_miss_tokens': 611
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e',
                'id': '96cfc003-238e-4be7-99bd-ab5a47167040',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a05c2c-17e2-7fc3-9d8c-507ac74c5640-0',
            tool_calls=[
                {
                    'name': 'search_customer_database',
                    'args': {'query': '李四'},
                    'id': 'call_00_sqSbTFw9HWdfYcqQA96z9761',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 611,
                'output_tokens': 40,
                'total_tokens': 651,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content='客户记录：李四，普通客户，最近购买日期：2025-12-20，累计消费：$3,200',
            name='search_customer_database',
            id='da2e7bb8-9934-4305-8448-e5f57a2c2dc8',
            tool_call_id='call_00_sqSbTFw9HWdfYcqQA96z9761'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 120,
                    'prompt_tokens': 697,
                    'total_tokens': 817,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 640},
                    'prompt_cache_hit_tokens': 640,
                    'prompt_cache_miss_tokens': 57
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e',
                'id': 'd5a31112-eb98-40d1-98d4-632a92fb39a7',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a05c2c-1c0c-73a3-bf6a-6e2fc0e54a31-0',
            tool_calls=[
                {
                    'name': 'CustomerAnalysis',
                    'args': {
                        'customer_name': '李四',
                        'customer_tier': '普通客户',
                        'recent_activity': '最近购买日期：2025-12-20',
                        'spending_level': '中',
                        'send_email': False
                    },
                    'id': 'call_00_9ebW3RiYtwmnblVHQcJM5313',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 697,
                'output_tokens': 120,
                'total_tokens': 817,
                'input_token_details': {'cache_read': 640},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content="Returning structured response: {'customer_name': '李四', 'customer_tier': '普通客户', 
'recent_activity': '最近购买日期：2025-12-20', 'spending_level': '中', 'send_email': False}",
            name='Custo

### 1.3 JsonSchema类型

举例1：

In [24]:
from typing import TypedDict, Annotated
from pydantic import BaseModel, Field
from langchain.agents.structured_output import ToolStrategy
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from rich import print as rprint

json_schema = {
    "title": "ContactInfo",
    "description": "用户的联系方式",
    "type": "object",
    "properties": {
        "name": {
            "description": "用户姓名",
            "type": "string"
        },
        "email": {
            "description": "用户邮箱地址",
            "type": "string"
        },
        "phone": {
            "description": "用户的手机号",
            "type": "string"
        }
    },
    "required": [
        "name",
        "email",
        "phone"
    ]
}

agent = create_agent(
    model=model,
    response_format=ToolStrategy(schema=json_schema)
)

response = agent.invoke({
    "messages" : [
        HumanMessage(content="从这段话中抽取结构化信息：小明的邮箱是shkstart@atguigu.com,电话是：13012341234")
    ]
})


rprint(response)

{
    'messages': [
        HumanMessage(
            content='从这段话中抽取结构化信息：小明的邮箱是shkstart@atguigu.com,电话是：13012341234',
            additional_kwargs={},
            response_metadata={},
            id='62bd9cc8-3ae6-4599-8193-d810925af706'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 78,
                    'prompt_tokens': 350,
                    'total_tokens': 428,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0},
                    'prompt_cache_hit_tokens': 0,
                    'prompt_cache_miss_tokens': 350
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e',
                'id': '4e50462b-5b36-4037-ba46-6318b67e287d',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a05c2c-20ff-77d3-835b-1019075feafa-0',
            tool_calls=[
                {
                    'name': 'ContactInfo',
                    'args': {'name': '小明', 'email': 'shkstart@atguigu.com', 'phone': '13012341234'},
                    'id': 'call_00_bvJ4nbHjquZRTf3tGf0c5953',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 350,
                'output_tokens': 78,
                'total_tokens': 428,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content="Returning structured response: {'name': '小明', 'email': 'shkstart@atguigu.com', 'phone': 
'13012341234'}",
            name='ContactInfo',
            id='af774047-4241-48ed-bd9e-bada399e1339',
            tool_call_id='call_00_bvJ4nbHjquZRTf3tGf0c5953'
        )
    ],
    'structured_response': {'name': '小明', 'email': 'shkstart@atguigu.com', 'phone': '13012341234'}
}

举例2：

In [25]:
from langchain_core.messages import SystemMessage
from pydantic import BaseModel, Field
from typing import Literal, Optional
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.tools import tool
from rich import print as rprint

# 定义工具
@tool(parse_docstring=True)
def search_customer_database(query: str) -> str:
    """
    在客户数据库中搜索信息

    Args:
        query (str): 客户查询字符串，例如 "张三" 或 "李四"

    Returns:
        str: 客户记录字符串，包含客户姓名、等级、最近购买日期和累计消费
    """
    # 模拟数据库查询结果
    if "张三" in query.lower():
        return "客户记录：张三，VIP客户，最近购买日期：2026-01-15，累计消费：$15,000"
    elif "李四" in query.lower():
        return "客户记录：李四，普通客户，最近购买日期：2025-12-20，累计消费：$3,200"
    else:
        return f"关于客户{query}，无记录"


@tool(parse_docstring=True)
def send_email(customer: str) -> str:
    """
    发送感谢邮件

    Args:
        customer (str): 客户名称，例如 "张三" 或 "李四"

    Returns:
        str: 确认消息，包含已发送的客户名称
    """
    return f"已向 {customer} 发送感谢邮件"


# 使用 json_schema 定义客户分析报告 Schema
customer_analysis_schema = {
    "title": "CustomerAnalysis",
    "type": "object",
    "description": "客户分析报告",
    "properties": {
        "customer_name": {
            "type": "string",
            "default": "",
            "description": "客户姓名"
        },
        "customer_tier": {
            "type": "string",
            "enum": ["潜在客户", "普通客户", "VIP客户", "流失风险"],
            "default": "潜在客户",
            "description": "客户等级"
        },
        "recent_activity": {
            "type": "string",
            "default": "",
            "description": "最近活动"
        },
        "spending_level": {
            "type": "string",
            "enum": ["低", "中", "高"],
            "default": "低",
            "description": "消费水平"
        },
        "send_email": {
            "type": "boolean",
            "default": False,
            "description": "是否已发送感谢邮件"
        }
    },
    # 所有字段都是必须输出的
    "required": ["customer_name", "customer_tier", "recent_activity", "spending_level"]
}

# 创建智能体
agent = create_agent(
    model=model,
    system_prompt=SystemMessage(content=""
                                        "请分析指定客户的情况："
                                        "1. 先搜索客户数据库了解最新情况 "
                                        "2. 如果是VIP客户，则发送感谢邮件 "
                                        "3. 基于搜索结果生成结构化分析报告 "
                                        "4. 如果用户提问与客户记录无关或找不到客户信息，则返回空对象，不发送感谢邮件"
                                ),
    tools=[search_customer_database, send_email],
    response_format=ToolStrategy(schema=customer_analysis_schema)
)

# 执行分析
result = agent.invoke({
    "messages": [{"role": "user", "content": "请分析客户张三"}]
    # "messages": [{"role": "user","content": "请分析客户李四"}]
    # "messages": [{"role": "user","content": "请分析客户王五"}]
    # "messages": [{"role": "user","content": "今天天气如何"}]
})

# 处理结果
rprint(result)

# if "structured_response" in result:
#     analysis = result["structured_response"]
#     print(analysis)

{
    'messages': [
        HumanMessage(
            content='请分析客户张三',
            additional_kwargs={},
            response_metadata={},
            id='cca84c01-26b6-49b5-98df-9534960b6e72'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 39,
                    'prompt_tokens': 631,
                    'total_tokens': 670,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 384},
                    'prompt_cache_hit_tokens': 384,
                    'prompt_cache_miss_tokens': 247
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e',
                'id': '54b2e25c-98ec-4d69-b92e-484e159a5675',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a05c2c-256d-7af1-8057-7de32e235df6-0',
            tool_calls=[
                {
                    'name': 'search_customer_database',
                    'args': {'query': '张三'},
                    'id': 'call_00_AOfmAIr1nRiKQGAmN4Ov3485',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 631,
                'output_tokens': 39,
                'total_tokens': 670,
                'input_token_details': {'cache_read': 384},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content='客户记录：张三，VIP客户，最近购买日期：2026-01-15，累计消费：$15,000',
            name='search_customer_database',
            id='d9c62f3f-1529-4a87-afaf-a30b40d8c7f8',
            tool_call_id='call_00_AOfmAIr1nRiKQGAmN4Ov3485'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 149,
                    'prompt_tokens': 715,
                    'total_tokens': 864,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 640},
                    'prompt_cache_hit_tokens': 640,
                    'prompt_cache_miss_tokens': 75
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e',
                'id': '6da05605-ab51-45de-b8c0-8f2e6c62b8e0',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a05c2c-27e9-7123-b461-eb8fe32a350b-0',
            tool_calls=[
                {
                    'name': 'send_email',
                    'args': {'customer': '张三'},
                    'id': 'call_00_0eV1bsSGwinUvFuI042C5304',
                    'type': 'tool_call'
                },
                {
                    'name': 'CustomerAnalysis',
                    'args': {
                        'customer_name': '张三',
                        'customer_tier': 'VIP客户',
                        'recent_activity': '最近购买日期：2026-01-15',
                        'spending_level': '高',
                        'send_email': True
                    },
                    'id': 'call_01_EPBWWZsHiS7YtnVCVSHX9733',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 715,
                'output_tokens': 149,
                'total_tokens': 864,
                'input_token_details': {'cache_read': 640},
                'output_token_details': {}
            }
      

### 1.4 @dataclass类型

举例1：

In [26]:
from pydantic import BaseModel, Field
from langchain.agents.structured_output import ToolStrategy
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from rich import print as rprint
from dataclasses import dataclass

@dataclass
class ContactInfo:
    """用户的联系方式"""
    name: str = Field(description="用户姓名")
    email: str = Field(description="用户邮箱地址")
    phone: str = Field(description="用户的手机号")


agent = create_agent(
    model=model,
    response_format=ToolStrategy(schema=ContactInfo)
)

response = agent.invoke({
    "messages" : [
        HumanMessage(content="从这段话中抽取结构化信息：小明的邮箱是shkstart@atguigu.com,电话是：13012341234")
    ]
})


rprint(response)

{
    'messages': [
        HumanMessage(
            content='从这段话中抽取结构化信息：小明的邮箱是shkstart@atguigu.com,电话是：13012341234',
            additional_kwargs={},
            response_metadata={},
            id='cfb85f2c-40d0-4163-a6dd-f742502dcb99'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 78,
                    'prompt_tokens': 350,
                    'total_tokens': 428,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 256},
                    'prompt_cache_hit_tokens': 256,
                    'prompt_cache_miss_tokens': 94
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e',
                'id': '090f9021-2813-4d3d-b7da-e4a87e1b0736',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a05c2c-2c0e-7003-978a-94e671a692d3-0',
            tool_calls=[
                {
                    'name': 'ContactInfo',
                    'args': {'name': '小明', 'email': 'shkstart@atguigu.com', 'phone': '13012341234'},
                    'id': 'call_00_Wb39mtRmtG9nw7wuvbjn3392',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 350,
                'output_tokens': 78,
                'total_tokens': 428,
                'input_token_details': {'cache_read': 256},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content="Returning structured response: ContactInfo(name='小明', email='shkstart@atguigu.com', 
phone='13012341234')",
            name='ContactInfo',
            id='c76637b9-ac76-4716-9590-aa614ee7bf22',
            tool_call_id='call_00_Wb39mtRmtG9nw7wuvbjn3392'
        )
    ],
    'structured_response': ContactInfo(name='小明', email='shkstart@atguigu.com', phone='13012341234')
}

举例2：

In [27]:
from langchain_core.messages import SystemMessage
from pydantic import BaseModel, Field
from typing import Literal
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.tools import tool
from rich import print as rprint

# 定义工具
@tool(parse_docstring=True)
def search_customer_database(query: str) -> str:
    """
    在客户数据库中搜索信息

    Args:
        query (str): 客户查询字符串，例如 "张三" 或 "李四"

    Returns:
        str: 客户记录字符串，包含客户姓名、等级、最近购买日期和累计消费
    """
    # 模拟数据库查询结果
    if "张三" in query.lower():
        return "客户记录：张三，VIP客户，最近购买日期：2026-01-15，累计消费：$15,000"
    elif "李四" in query.lower():
        return "客户记录：李四，普通客户，最近购买日期：2025-12-20，累计消费：$3,200"
    else:
        return f"关于客户{query}，无记录"


@tool(parse_docstring=True)
def send_email(customer: str) -> str:
    """
    发送感谢邮件

    Args:
        customer (str): 客户名称，例如 "张三" 或 "李四"

    Returns:
        str: 确认消息，包含已发送的客户名称
    """
    return f"已向 {customer} 发送感谢邮件"


# 定义dataclass
@dataclass
class CustomerAnalysis:
    """客户分析报告"""
    customer_name: str = Field(None, description="客户姓名")
    customer_tier: Literal["潜在客户", "普通客户", "VIP客户", "流失风险"] = Field("潜在客户",description="客户等级,只能是潜在客户、普通客户、VIP客户或流失风险")
    recent_activity: str = Field(None, description="最近活动")
    spending_level: Literal["低", "中", "高"] = Field(None, description="消费水平")
    send_email: bool = Field(False, description="是否已发送感谢邮件")


# 创建智能体
agent = create_agent(
    model=model,
    system_prompt=SystemMessage(content=""
                                        "请分析指定客户的情况："
                                        "1. 先搜索客户数据库了解最新情况 "
                                        "2. 如果是VIP客户，则发送感谢邮件 "
                                        "3. 基于搜索结果生成结构化分析报告 "
                                        "4. 如果用户提问与客户记录无关或找不到客户信息，则返回空对象，不发送感谢邮件"
                                ),
    tools=[search_customer_database, send_email],
    response_format=ToolStrategy(schema=CustomerAnalysis)
)

# 执行分析
result = agent.invoke({
    # "messages": [{"role": "user", "content": "请分析客户张三"}]
    "messages": [{"role": "user","content": "请分析客户李四"}]
    # "messages": [{"role": "user","content": "请分析客户王五"}]
    # "messages": [{"role": "user","content": "今天天气如何"}]
})

# 处理结果
rprint(result)

# if "structured_response" in result:
#     analysis = result["structured_response"]
#     print(analysis)

{
    'messages': [
        HumanMessage(
            content='请分析客户李四',
            additional_kwargs={},
            response_metadata={},
            id='1abe55aa-5650-4c81-8384-8a8847f47163'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 40,
                    'prompt_tokens': 623,
                    'total_tokens': 663,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 512},
                    'prompt_cache_hit_tokens': 512,
                    'prompt_cache_miss_tokens': 111
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e',
                'id': '88aac80e-f778-4e41-8093-2f02c8119a6d',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a05c2c-3121-7ae0-9099-4834ac984f5e-0',
            tool_calls=[
                {
                    'name': 'search_customer_database',
                    'args': {'query': '李四'},
                    'id': 'call_00_w7awI2eOJSMXId7OvWzP9721',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 623,
                'output_tokens': 40,
                'total_tokens': 663,
                'input_token_details': {'cache_read': 512},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content='客户记录：李四，普通客户，最近购买日期：2025-12-20，累计消费：$3,200',
            name='search_customer_database',
            id='ecb36739-1c71-44b4-b123-cd12f9ea3c1f',
            tool_call_id='call_00_w7awI2eOJSMXId7OvWzP9721'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 120,
                    'prompt_tokens': 709,
                    'total_tokens': 829,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 640},
                    'prompt_cache_hit_tokens': 640,
                    'prompt_cache_miss_tokens': 69
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e',
                'id': '343d32e2-d44e-46cf-8e0b-3d6550d6f155',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a05c2c-34b4-73e3-a367-6421c5e63ad6-0',
            tool_calls=[
                {
                    'name': 'CustomerAnalysis',
                    'args': {
                        'customer_name': '李四',
                        'customer_tier': '普通客户',
                        'recent_activity': '最近购买日期：2025-12-20',
                        'spending_level': '中',
                        'send_email': False
                    },
                    'id': 'call_00_VfGgttQEXmfAJrctRDGA3960',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 709,
                'output_tokens': 120,
                'total_tokens': 829,
                'input_token_details': {'cache_read': 640},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content="Returning structured response: CustomerAnalysis(customer_name='李四', 
customer_tier='普通客户', recent_activity='最近购买日期：2025-12-20', spending_level='中', send_email=False)",
            name

### 1.5 多schema联合模式

举例

In [34]:
from pydantic import BaseModel, Field
from typing import Union
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.messages import HumanMessage


class ContactInfo(BaseModel):
    """用户的联系方式"""
    name: str = Field(description="用户姓名")
    email: str = Field(description="用户邮箱地址")
    phone: str = Field(description="用户的手机号")

class EventInfo(BaseModel):
    """事件详情"""
    event_name: str = Field(description="事件名称")
    date: str = Field(description="事件发生日期")

agent = create_agent(
    model=model,
    response_format=ToolStrategy(schema=Union[ContactInfo, EventInfo])
)

response = agent.invoke(
    {
        "messages": [
            HumanMessage("从这段话中抽取结构化信息：小明的邮箱地址为：shkstart@atguigu.com，手机号：12345678912")
        ]
    }
)

# for msg in response["messages"]:
#     msg.pretty_print()

rprint(response)

{
    'messages': [
        HumanMessage(
            content='从这段话中抽取结构化信息：小明的邮箱地址为：shkstart@atguigu.com，手机号：12345678912',
            additional_kwargs={},
            response_metadata={},
            id='3f2e1379-f149-441c-abf1-64038403d07b'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 78,
                    'prompt_tokens': 424,
                    'total_tokens': 502,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 384},
                    'prompt_cache_hit_tokens': 384,
                    'prompt_cache_miss_tokens': 40
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e',
                'id': '6861ea5e-dcee-4422-b5ef-6950c872fc09',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a05c2e-dfa1-76b1-9938-325e2455853f-0',
            tool_calls=[
                {
                    'name': 'ContactInfo',
                    'args': {'name': '小明', 'email': 'shkstart@atguigu.com', 'phone': '12345678912'},
                    'id': 'call_00_6V6dotbrxhAoi4k1gLTG0661',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 424,
                'output_tokens': 78,
                'total_tokens': 502,
                'input_token_details': {'cache_read': 384},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content="Returning structured response: name='小明' email='shkstart@atguigu.com' phone='12345678912'",
            name='ContactInfo',
            id='38fb09e2-856c-45ff-8611-e7d625efaec8',
            tool_call_id='call_00_6V6dotbrxhAoi4k1gLTG0661'
        )
    ],
    'structured_response': ContactInfo(name='小明', email='shkstart@atguigu.com', phone='12345678912')
}

In [29]:
response = agent.invoke(
    {
        "messages": [
            HumanMessage("从这段话中抽取结构化信息：2026年高考报名人数突破1200万")
        ]
    }
)

for msg in response["messages"]:
    msg.pretty_print()

print(response["structured_response"])

================================ Human Message =================================

从这段话中抽取结构化信息：2026年高考报名人数突破1200万
================================== Ai Message ==================================
Tool Calls:
  EventInfo (call_00_cITc2ccAoUZpRt0XbNlW2126)
 Call ID: call_00_cITc2ccAoUZpRt0XbNlW2126
  Args:
    event_name: 高考报名
    date: 2026年
================================= Tool Message =================================
Name: EventInfo

Returning structured response: event_name='高考报名' date='2026年'
event_name='高考报名' date='2026年'


## 2、自定义工具消息：tool_message_content参数

举例：

In [30]:
from pydantic import BaseModel, Field
from langchain.agents.structured_output import ToolStrategy
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from rich import print as rprint

class ContactInfo(BaseModel):
    """用户的联系方式"""
    name: str = Field(description="用户姓名")
    email: str = Field(description="用户邮箱地址")
    phone: str = Field(description="用户的手机号")


agent = create_agent(
    model=model,
    response_format=ToolStrategy(ContactInfo)
)

response = agent.invoke({
        "messages": [
            HumanMessage("从这段话中抽取结构化信息：小明的邮箱地址为：songhk@atguigu.com，手机号：12345678912")
        ]
    }
)

rprint(response)

{
    'messages': [
        HumanMessage(
            content='从这段话中抽取结构化信息：小明的邮箱地址为：songhk@atguigu.com，手机号：12345678912',
            additional_kwargs={},
            response_metadata={},
            id='7b319515-04b7-4a02-95c7-ca2558b93fe0'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 77,
                    'prompt_tokens': 351,
                    'total_tokens': 428,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 256},
                    'prompt_cache_hit_tokens': 256,
                    'prompt_cache_miss_tokens': 95
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e',
                'id': 'c1db5003-9080-439e-9553-b687ca05e59d',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a05c2c-3fe0-73e0-8cb6-a5ab20a014af-0',
            tool_calls=[
                {
                    'name': 'ContactInfo',
                    'args': {'name': '小明', 'email': 'songhk@atguigu.com', 'phone': '12345678912'},
                    'id': 'call_00_i8Wn9vlHlsJYNm7LwzHg7177',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 351,
                'output_tokens': 77,
                'total_tokens': 428,
                'input_token_details': {'cache_read': 256},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content="Returning structured response: name='小明' email='songhk@atguigu.com' phone='12345678912'",
            name='ContactInfo',
            id='6a33fca0-dc35-4177-9515-4c808e9b379c',
            tool_call_id='call_00_i8Wn9vlHlsJYNm7LwzHg7177'
        )
    ],
    'structured_response': ContactInfo(name='小明', email='songhk@atguigu.com', phone='12345678912')
}

作为对比：

In [31]:
from pydantic import BaseModel, Field
from langchain.agents.structured_output import ToolStrategy
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from rich import print as rprint

class ContactInfo(BaseModel):
    """用户的联系方式"""
    name: str = Field(description="用户姓名")
    email: str = Field(description="用户邮箱地址")
    phone: str = Field(description="用户的手机号")


agent = create_agent(
    model=model,
    response_format=ToolStrategy(
        schema = ContactInfo,
        tool_message_content="已成功抽取信息"
    )
)

response = agent.invoke({
        "messages": [
            HumanMessage("从这段话中抽取结构化信息：小明的邮箱地址为：songhk@atguigu.com，手机号：12345678912")
        ]
    }
)

rprint(response)

{
    'messages': [
        HumanMessage(
            content='从这段话中抽取结构化信息：小明的邮箱地址为：songhk@atguigu.com，手机号：12345678912',
            additional_kwargs={},
            response_metadata={},
            id='2f8fb301-591a-4278-ad4f-7d525a0f5870'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 77,
                    'prompt_tokens': 351,
                    'total_tokens': 428,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 256},
                    'prompt_cache_hit_tokens': 256,
                    'prompt_cache_miss_tokens': 95
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e',
                'id': 'b6ce14ab-0f0f-48ad-9e81-2aa84093bf4b',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a05c2c-4486-7f42-8c61-fdca2a26983d-0',
            tool_calls=[
                {
                    'name': 'ContactInfo',
                    'args': {'name': '小明', 'email': 'songhk@atguigu.com', 'phone': '12345678912'},
                    'id': 'call_00_I9v9OrG6Qn0MXlp0qRTN3282',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 351,
                'output_tokens': 77,
                'total_tokens': 428,
                'input_token_details': {'cache_read': 256},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content='已成功抽取信息',
            name='ContactInfo',
            id='bb86989e-b31e-47aa-a86b-c5cf1acae5dd',
            tool_call_id='call_00_I9v9OrG6Qn0MXlp0qRTN3282'
        )
    ],
    'structured_response': ContactInfo(name='小明', email='songhk@atguigu.com', phone='12345678912')
}